# Train A Piper Voice In Colab

This notebook is the public quick-start path for the repository.

Workflow:

1. mount Google Drive
2. point the notebook at a reviewed dataset
3. train in Colab
4. export ONNX
5. copy the results back to Drive

## 0. Check Colab GPU

In [ ]:
import sys
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
!nvidia-smi


## 1. Install system packages

In [ ]:
!sudo apt-get update -y
!sudo apt-get install -y build-essential cmake ninja-build espeak-ng espeak-ng-data libespeak-ng-dev pkg-config ffmpeg
!pkg-config --modversion espeak-ng


## 2. Clone Piper and install training dependencies

In [ ]:
%cd /content
!rm -rf piper1-gpl
!git clone https://github.com/OHF-Voice/piper1-gpl.git
%cd /content/piper1-gpl
!python3 -m pip install --upgrade pip setuptools wheel scikit-build cmake ninja
!python3 -m pip install -e ".[train]"
!chmod +x ./build_monotonic_align.sh
!./build_monotonic_align.sh


## 3. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 4. Set project paths and training values

Edit this section only. Everything below uses these variables.


In [ ]:
!pip -q install huggingface_hub hf_xet


In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download

VOICE_NAME = "my_voice"
ESPEAK_VOICE = "ru"
SAMPLE_RATE_HZ = 22050
BATCH_SIZE = 8
EXTRA_EPOCHS = 500  # Additional fine-tuning epochs on top of the base checkpoint.

PROJECT_DIR = Path("/content/drive/MyDrive/sherpa_tts/data/my_voice")
METADATA_PATH = PROJECT_DIR / "metadata.csv"
AUDIO_DIR = PROJECT_DIR / "wavs"

CACHE_DIR = Path("/content/piper_cache")
RUN_DIR = Path("/content/drive/MyDrive/sherpa_tts/runs") / VOICE_NAME
EXPORT_DIR = Path("/content/drive/MyDrive/sherpa_tts/exports") / VOICE_NAME
CHECKPOINT_CONFIG_PATH = RUN_DIR / "checkpoint_config.yaml"
CONFIG_PATH = RUN_DIR / f"{VOICE_NAME}.json"

CKPT_ROOT = Path("/content/drive/MyDrive/sherpa_tts/base_checkpoints")
BASE_REPO_ID = "rhasspy/piper-checkpoints"
BASE_CKPT_GLOB = "ru/ru_RU/irina/medium/*"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_ROOT.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id=BASE_REPO_ID,
    repo_type="dataset",
    allow_patterns=[BASE_CKPT_GLOB],
    local_dir=str(CKPT_ROOT),
)

checkpoint_candidates = sorted(CKPT_ROOT.glob(BASE_CKPT_GLOB))
assert checkpoint_candidates, f"No checkpoints matched {BASE_CKPT_GLOB} under {CKPT_ROOT}"
BASE_CHECKPOINT = str(next(path for path in checkpoint_candidates if path.suffix == ".ckpt"))
TRAIN_MAX_EPOCHS = None

print("Project dir:", PROJECT_DIR)
print("Metadata:", METADATA_PATH)
print("Audio dir:", AUDIO_DIR)
print("Config path:", CONFIG_PATH)
print("Export dir:", EXPORT_DIR)
print("Run dir:", RUN_DIR)
print("Base checkpoint:", BASE_CHECKPOINT)
print("Extra epochs:", EXTRA_EPOCHS)


## 5. Validate the dataset


In [ ]:
import pandas as pd

assert METADATA_PATH.is_file(), f"metadata.csv not found: {METADATA_PATH}"
assert AUDIO_DIR.is_dir(), f"wavs directory not found: {AUDIO_DIR}"

df = pd.read_csv(METADATA_PATH, sep="|", header=None, names=["clip_id", "text"])
assert not df.empty, "metadata.csv is empty"

def resolve_audio_path(clip_id: str) -> Path:
    stem = str(clip_id).strip()
    with_suffix = AUDIO_DIR / stem
    if with_suffix.exists():
        return with_suffix
    return AUDIO_DIR / f"{stem}.wav"

missing = [str(resolve_audio_path(row.clip_id)) for row in df.itertuples() if not resolve_audio_path(row.clip_id).exists()]

print(df.head())
print("Rows:", len(df))
print("Missing audio files:", len(missing))
if missing:
    print("First missing file:", missing[0])

assert not missing, "Some audio files referenced by metadata.csv are missing"


## 6. Write checkpoint config


In [ ]:
checkpoint_config = """trainer:
  callbacks:
    - class_path: lightning.pytorch.callbacks.ModelCheckpoint
      init_args:
        every_n_epochs: 5
        save_top_k: 1
        save_last: true
"""
CHECKPOINT_CONFIG_PATH.write_text(checkpoint_config, encoding="utf-8")
print(CHECKPOINT_CONFIG_PATH.read_text(encoding="utf-8"))


## 7. Train


In [ ]:
import torch

src = Path(BASE_CHECKPOINT)
dst = src.with_name(src.stem + "-clean.ckpt")

ckpt = torch.load(src, map_location="cpu", weights_only=False)
base_epoch = int(ckpt.get("epoch", -1))
base_step = int(ckpt.get("global_step", -1))

print("Base checkpoint epoch:", base_epoch)
print("Base checkpoint global_step:", base_step)
print("Has hyper_parameters:", "hyper_parameters" in ckpt)

ckpt.pop("hyper_parameters", None)
torch.save(ckpt, dst)

BASE_CHECKPOINT = str(dst)
TRAIN_MAX_EPOCHS = EXTRA_EPOCHS if base_epoch < 0 else base_epoch + EXTRA_EPOCHS
print("Clean checkpoint:", BASE_CHECKPOINT)
print("Training will stop at epoch:", TRAIN_MAX_EPOCHS)


In [ ]:
import subprocess

train_cmd = [
    "python3",
    "-m",
    "piper.train",
    "fit",
    "--config",
    str(CHECKPOINT_CONFIG_PATH),
    "--trainer.default_root_dir",
    str(RUN_DIR),
    "--trainer.max_epochs",
    str(TRAIN_MAX_EPOCHS),
    "--data.voice_name",
    VOICE_NAME,
    "--data.csv_path",
    str(METADATA_PATH),
    "--data.audio_dir",
    str(AUDIO_DIR),
    "--model.sample_rate",
    str(SAMPLE_RATE_HZ),
    "--data.espeak_voice",
    ESPEAK_VOICE,
    "--data.cache_dir",
    str(CACHE_DIR),
    "--data.config_path",
    str(CONFIG_PATH),
    "--data.batch_size",
    str(BATCH_SIZE),
]

if BASE_CHECKPOINT.strip():
    train_cmd.extend(["--ckpt_path", BASE_CHECKPOINT.strip()])

print(" ".join(train_cmd))
result = subprocess.run(
    train_cmd,
    cwd="/content/piper1-gpl",
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("--- STDERR ---")
    print(result.stderr)
    raise subprocess.CalledProcessError(result.returncode, train_cmd)


## 8. Export ONNX from the latest checkpoint


In [ ]:
checkpoints = sorted(RUN_DIR.rglob("*.ckpt"), key=lambda path: path.stat().st_mtime)
assert checkpoints, f"No checkpoints found under {RUN_DIR}"

latest_checkpoint = checkpoints[-1]
onnx_path = EXPORT_DIR / "model.onnx"

print("Latest checkpoint:", latest_checkpoint)
print("Output ONNX:", onnx_path)


In [ ]:
import json
import onnx
import shutil
import subprocess

export_cmd = [
    "python3",
    "-m",
    "piper.train.export_onnx",
    "--checkpoint",
    str(latest_checkpoint),
    "--output-file",
    str(onnx_path),
]

print(" ".join(export_cmd))
subprocess.run(export_cmd, cwd="/content/piper1-gpl", check=True)

config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
tokens_path = EXPORT_DIR / "tokens.txt"
phoneme_id_map = config["phoneme_id_map"]

with tokens_path.open("w", encoding="utf-8", newline="\n") as tokens_file:
    for symbol, ids in phoneme_id_map.items():
        value = ids[0] if isinstance(ids, list) else ids
        tokens_file.write(f"{symbol} {value}\n")

metadata = {
    "model_type": "vits",
    "comment": "piper",
    "language": config.get("language", {}).get("name_english") or config["espeak"]["voice"],
    "voice": config["espeak"]["voice"],
    "has_espeak": int(config.get("phoneme_type") == "espeak"),
    "n_speakers": int(config["num_speakers"]),
    "sample_rate": int(config["audio"]["sample_rate"]),
}

model = onnx.load(str(onnx_path))
existing = {item.key: item for item in model.metadata_props}
for key, value in metadata.items():
    if key in existing:
        existing[key].value = str(value)
    else:
        meta = model.metadata_props.add()
        meta.key = key
        meta.value = str(value)
onnx.save(model, str(onnx_path))

espeak_data_src = Path("/content/piper1-gpl/src/piper/espeak-ng-data")
if espeak_data_src.is_dir():
    shutil.copytree(espeak_data_src, EXPORT_DIR / "espeak-ng-data", dirs_exist_ok=True)
    print("Copied espeak-ng-data from:", espeak_data_src)
else:
    print("WARNING: espeak-ng-data was not found under", espeak_data_src)

print("Generated tokens:", tokens_path)
print("Added metadata:", metadata)


## 9. Save export artifacts


In [ ]:
import shutil

assert CONFIG_PATH.is_file(), f"Config json not found: {CONFIG_PATH}"
checkpoint_copy = EXPORT_DIR / latest_checkpoint.name
config_copy = EXPORT_DIR / CONFIG_PATH.name
shutil.copy2(latest_checkpoint, checkpoint_copy)
shutil.copy2(CONFIG_PATH, config_copy)
print("Copied checkpoint:", checkpoint_copy)
print("Copied config:", config_copy)
print("Export directory contents:")
for path in sorted(EXPORT_DIR.iterdir()):
    print("-", path.name)


## Notes

- This notebook intentionally removes saved outputs and personal hardcoded paths.
- `EXTRA_EPOCHS` means additional fine-tuning on top of the base checkpoint epoch, so you do not need to manually add the base epoch yourself.
- The base checkpoint cell strips old `hyper_parameters` before training to avoid LightningCLI checkpoint parsing issues with older Piper checkpoints.
- The export cell now writes `tokens.txt`, embeds Sherpa metadata into `model.onnx`, and copies `espeak-ng-data` for a speak-ready bundle.
